# 02 · Can free Colab train it?

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/CSroseX/Legal-Llama/blob/main/notebooks/02_training_feasibility.ipynb)

Runtime → Change runtime type → **T4 GPU**, then Runtime → **Run all**. Paste the output of the last cell back to Claude.

In [ ]:
!pip install -q unsloth

In [ ]:
from unsloth import FastLanguageModel
import torch, time, gc, json
assert torch.cuda.is_available(), "No GPU. Kaggle: right panel > Session options > Accelerator > GPU T4 x2 (and Internet: On). Colab: Runtime > Change runtime type > T4 GPU"
gpu = torch.cuda.get_device_properties(0)
report = {"gpu": gpu.name, "gpu_memory_gb": round(gpu.total_memory / 1e9, 1)}
print(report)

In [ ]:
import pandas as pd
from huggingface_hub import hf_hub_download, list_repo_files

DATA_REPO = "theatticusproject/cuad-qa"
DATA_REV = "refs/convert/parquet"
paths = [
    hf_hub_download(DATA_REPO, f, repo_type="dataset", revision=DATA_REV)
    for f in list_repo_files(DATA_REPO, repo_type="dataset", revision=DATA_REV)
    if f.startswith("default/train/") and f.endswith(".parquet")
]
train = pd.concat([pd.read_parquet(p) for p in paths], ignore_index=True)
train["first_answer"] = train["answers"].apply(lambda a: a["text"][0] if len(a["text"]) else "")
report["train_rows"] = len(train)
print(report["train_rows"])

In [ ]:
BASE = "unsloth/llama-3-8b-bnb-4bit"
LENGTHS = [1024, 2048, 4096]

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE,
    max_seq_length=max(LENGTHS),
    load_in_4bit=True,
    dtype=None,
)
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    bias="none",
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    use_gradient_checkpointing="unsloth",
    random_state=3407,
)
trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
report["trainable_params"] = trainable
report["trainable_share_percent"] = round(100 * trainable / total, 3)
report["memory_after_load_gb"] = round(torch.cuda.memory_reserved() / 1e9, 1)
print(report)

In [ ]:
contracts = train.drop_duplicates("title")[["title", "context"]]
contract_ids = {
    t: tokenizer(c, add_special_tokens=False)["input_ids"]
    for t, c in zip(contracts.title, contracts.context)
}
q_tokens = train.question.map(lambda q: len(tokenizer(q, add_special_tokens=False)["input_ids"]))
a_tokens = train.first_answer.map(lambda a: len(tokenizer(a, add_special_tokens=False)["input_ids"]))
full_tokens = train.title.map(lambda t: len(contract_ids[t])) + q_tokens + a_tokens + 60

def windows(length, count):
    long_ones = [ids for ids in contract_ids.values() if len(ids) >= length]
    return [[tokenizer.bos_token_id] + long_ones[i % len(long_ones)][: length - 1] for i in range(count)]

print(len(contract_ids), "contracts tokenized")

In [ ]:
import bitsandbytes as bnb

optimizer = bnb.optim.AdamW8bit([p for p in model.parameters() if p.requires_grad], lr=2e-4)
FastLanguageModel.for_training(model)
WARMUP, MEASURE = 2, 8

def measure(length):
    batches = windows(length, WARMUP + MEASURE)
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    for i, ids in enumerate(batches):
        if i == WARMUP:
            torch.cuda.synchronize()
            start = time.time()
        x = torch.tensor([ids], device="cuda")
        loss = model(input_ids=x, labels=x).loss
        loss.backward()
        optimizer.step()
        optimizer.zero_grad(set_to_none=True)
    torch.cuda.synchronize()
    seconds = (time.time() - start) / MEASURE
    return {
        "fits": True,
        "seconds_per_step": round(seconds, 2),
        "tokens_per_second": round(length / seconds),
        "peak_memory_gb": round(torch.cuda.max_memory_reserved() / 1e9, 1),
        "last_loss": round(loss.item(), 3),
    }

report["speed"] = {}
for length in LENGTHS:
    try:
        report["speed"][length] = measure(length)
    except torch.cuda.OutOfMemoryError:
        optimizer.zero_grad(set_to_none=True)
        gc.collect()
        torch.cuda.empty_cache()
        report["speed"][length] = {"fits": False}
    print(length, report["speed"][length])

In [ ]:
report["one_epoch_estimate_hours"] = {}
for length, result in report["speed"].items():
    if not result["fits"]:
        continue
    tokens = int(full_tokens.clip(upper=length).sum())
    hours = tokens / result["tokens_per_second"] / 3600
    report["one_epoch_estimate_hours"][length] = {
        "epoch_tokens_millions": round(tokens / 1e6, 1),
        "all_data": round(hours, 1),
        "half_data": round(hours / 2, 1),
        "quarter_data": round(hours / 4, 1),
    }
print(json.dumps(report, indent=1))